In [1]:
import sys 
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data" 




from utilities.model import CNNmodel_base, CNNModel1
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

In [2]:
import pandas as pd

In [3]:
df=pd.read_csv(DATA_DIR / 'processed_data.csv')

In [4]:
inputs=[
        
        'acc_x_u1', 'acc_x_u2', 'acc_x_u3', 'acc_x_u4', 'acc_x_u5', 
        'acc_y_u1', 'acc_y_u2', 'acc_y_u3', 'acc_y_u4', 'acc_y_u5', 
        'acc_z_u1', 'acc_z_u2', 'acc_z_u3', 'acc_z_u4', 'acc_z_u5',
        'gyr_x_u1', 'gyr_x_u2', 'gyr_x_u3', 'gyr_x_u4', 'gyr_x_u5', 
        'gyr_y_u1', 'gyr_y_u2', 'gyr_y_u3', 'gyr_y_u4', 'gyr_y_u5',
        'gyr_z_u1', 'gyr_z_u2', 'gyr_z_u3', 'gyr_z_u4', 'gyr_z_u5', 
        'mag_x_u1', 'mag_x_u2', 'mag_x_u3', 'mag_x_u4', 'mag_x_u5', 
        'mag_y_u1', 'mag_y_u2', 'mag_y_u3', 'mag_y_u4', 'mag_y_u5', 
        'mag_z_u1', 'mag_z_u2', 'mag_z_u3', 'mag_z_u4', 'mag_z_u5',
]


In [5]:
exercises = ['e1','e2','e3','e4','e5','e6','e7','e8']
subject=['s1', 's2', 's3','s4', 's5']
labels=['correct', 'low_amplitude', 'fast']

for s in  subject:
    for e in exercises:
        for l in labels:


            df_r=df[(df['subject'] == s) & (df['exercise'] == e) & (df['label']==l)]

            
            duration = df_r['time index'].max() - df_r['time index'].min()

            repetition_duration= duration /10

            print(repetition_duration, duration, l)

            

202.8 2028 correct
192.3 1923 low_amplitude
84.2 842 fast
192.8 1928 correct
189.3 1893 low_amplitude
125.9 1259 fast
197.5 1975 correct
202.4 2024 low_amplitude
109.1 1091 fast
204.2 2042 correct
173.6 1736 low_amplitude
101.4 1014 fast
182.1 1821 correct
185.7 1857 low_amplitude
91.2 912 fast
237.1 2371 correct
189.3 1893 low_amplitude
110.4 1104 fast
224.8 2248 correct
198.4 1984 low_amplitude
104.9 1049 fast
180.9 1809 correct
191.7 1917 low_amplitude
84.5 845 fast
157.8 1578 correct
147.5 1475 low_amplitude
82.4 824 fast
184.3 1843 correct
153.7 1537 low_amplitude
100.8 1008 fast
171.5 1715 correct
136.7 1367 low_amplitude
82.9 829 fast
146.2 1462 correct
124.4 1244 low_amplitude
84.1 841 fast
139.1 1391 correct
116.9 1169 low_amplitude
81.5 815 fast
144.3 1443 correct
120.0 1200 low_amplitude
78.4 784 fast
131.8 1318 correct
108.9 1089 low_amplitude
77.5 775 fast
120.4 1204 correct
106.8 1068 low_amplitude
74.6 746 fast
164.3 1643 correct
149.3 1493 low_amplitude
85.8 858 fast
17

In [6]:
label_to_idx = {
    'correct': 0,
    'low_amplitude': 1,
    'fast': 2
}

all_sequences = []
all_labels = []
all_subjects = []   
all_exercises = []  

for s in subject:
    for e in exercises:
        for l in labels:

            df_r = df[
                (df['subject'] == s) &
                (df['exercise'] == e) &
                (df['label'] == l)
            ].copy()

            df_r = df_r.sort_values('time index')

            duration = df_r['time index'].max() - df_r['time index'].min()
            rep_len = int(duration / 10)

            for i in range(10):
                rep = df_r.iloc[i * rep_len:(i + 1) * rep_len]

                X_rep = rep[inputs].to_numpy()   # (timesteps, features)

                all_sequences.append(X_rep)
                all_labels.append(label_to_idx[l])
                all_subjects.append(s)
                all_exercises.append(e)

max_len = max(seq.shape[0] for seq in all_sequences)

padded = []

for seq in all_sequences:
    seq = torch.tensor(seq, dtype=torch.float32)  # (time, features)
    seq = seq.transpose(0, 1)                     # (features, time)

    pad_len = max_len - seq.shape[1]
    seq = F.pad(seq, (0, pad_len))

    padded.append(seq)

X = torch.stack(padded)  # (N, features, max_len)
y = torch.tensor(all_labels, dtype=torch.long)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [8]:
all_subjects = np.array(all_subjects)
unique_subjects = np.unique(all_subjects)
fold_results = []

print(type(all_subjects))
print(np.array(all_subjects).shape)
print(all_subjects[:10] if hasattr(all_subjects, "__len__") else all_subjects)
print("len(X) =", len(X))
print("len(y) =", len(y))
print("len(all_subjects) =", len(all_subjects) if hasattr(all_subjects, "__len__") else "NO LEN")


<class 'numpy.ndarray'>
(1200,)
['s1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1' 's1']
len(X) = 1200
len(y) = 1200
len(all_subjects) = 1200


In [9]:


for val_subject in unique_subjects:
    print(f"\n========== Fold: leaving out {val_subject} ==========")

    train_indices = [i for i, s in enumerate(all_subjects) if s != val_subject]
    val_indices = [i for i, s in enumerate(all_subjects) if s == val_subject]

    X_train = X[train_indices]
    y_train = y[train_indices]
    X_val = X[val_indices]
    y_val = y[val_indices]

    # convert to tensors
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    # fresh model for each fold
    model = CNNModel1(input_dim=45, num_classes=3).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val_loss = float("inf")
    best_preds = None
    best_labels = None

    for epoch in range(20):
        # TRAIN
        model.train()
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        train_loss = running_loss / len(train_loader.dataset)

        # VALIDATION
        model.eval()
        val_running_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * inputs.size(0)

                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(all_labels, all_preds)

        print(
            f"Fold {val_subject} | Epoch {epoch+1} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

        # keep best epoch for this fold
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_preds = np.array(all_preds)
            best_labels = np.array(all_labels)

    # fold-level metrics
    fold_acc = accuracy_score(best_labels, best_preds)
    fold_f1 = f1_score(best_labels, best_preds, average="macro")
    fold_cm = confusion_matrix(best_labels, best_preds)

    fold_results.append({
        "subject": val_subject,
        "val_loss": best_val_loss,
        "accuracy": fold_acc,
        "macro_f1": fold_f1,
        "confusion_matrix": fold_cm
    })

    print(f"\nBest results for held-out subject {val_subject}:")
    print(f"Val Loss: {best_val_loss:.4f}")
    print(f"Accuracy: {fold_acc:.4f}")
    print(f"Macro F1: {fold_f1:.4f}")
    print("Confusion Matrix:")
    print(fold_cm)


========== Fold: leaving out s1 ==========


C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\

Fold s1 | Epoch 1 | Train Loss: 0.9141 | Val Loss: 0.8628 | Val Acc: 0.6417
Fold s1 | Epoch 2 | Train Loss: 0.5472 | Val Loss: 1.0244 | Val Acc: 0.5917
Fold s1 | Epoch 3 | Train Loss: 0.3231 | Val Loss: 1.0897 | Val Acc: 0.6458
Fold s1 | Epoch 4 | Train Loss: 0.1732 | Val Loss: 1.2996 | Val Acc: 0.6333
Fold s1 | Epoch 5 | Train Loss: 0.1538 | Val Loss: 1.4895 | Val Acc: 0.6500
Fold s1 | Epoch 6 | Train Loss: 0.1398 | Val Loss: 1.7502 | Val Acc: 0.6042
Fold s1 | Epoch 7 | Train Loss: 0.0742 | Val Loss: 2.1030 | Val Acc: 0.6042
Fold s1 | Epoch 8 | Train Loss: 0.0479 | Val Loss: 2.5213 | Val Acc: 0.5125
Fold s1 | Epoch 9 | Train Loss: 0.0734 | Val Loss: 1.6339 | Val Acc: 0.7083
Fold s1 | Epoch 10 | Train Loss: 0.1351 | Val Loss: 2.2504 | Val Acc: 0.4958
Fold s1 | Epoch 11 | Train Loss: 0.0921 | Val Loss: 2.5425 | Val Acc: 0.4833
Fold s1 | Epoch 12 | Train Loss: 0.0522 | Val Loss: 2.4064 | Val Acc: 0.6667
Fold s1 | Epoch 13 | Train Loss: 0.0450 | Val Loss: 2.1321 | Val Acc: 0.6875
Fold s1 

C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\

Fold s2 | Epoch 1 | Train Loss: 0.9526 | Val Loss: 0.7758 | Val Acc: 0.6333
Fold s2 | Epoch 2 | Train Loss: 0.6572 | Val Loss: 0.6722 | Val Acc: 0.6333
Fold s2 | Epoch 3 | Train Loss: 0.4239 | Val Loss: 0.5296 | Val Acc: 0.7250
Fold s2 | Epoch 4 | Train Loss: 0.3010 | Val Loss: 0.5880 | Val Acc: 0.7917
Fold s2 | Epoch 5 | Train Loss: 0.2334 | Val Loss: 0.6175 | Val Acc: 0.7417
Fold s2 | Epoch 6 | Train Loss: 0.1543 | Val Loss: 0.4827 | Val Acc: 0.8250
Fold s2 | Epoch 7 | Train Loss: 0.1486 | Val Loss: 0.9868 | Val Acc: 0.6708
Fold s2 | Epoch 8 | Train Loss: 0.1221 | Val Loss: 0.6413 | Val Acc: 0.8458
Fold s2 | Epoch 9 | Train Loss: 0.1041 | Val Loss: 0.6983 | Val Acc: 0.7708
Fold s2 | Epoch 10 | Train Loss: 0.0684 | Val Loss: 0.6170 | Val Acc: 0.8458
Fold s2 | Epoch 11 | Train Loss: 0.1080 | Val Loss: 0.8437 | Val Acc: 0.7708
Fold s2 | Epoch 12 | Train Loss: 0.1028 | Val Loss: 0.9927 | Val Acc: 0.7625
Fold s2 | Epoch 13 | Train Loss: 0.1128 | Val Loss: 0.6695 | Val Acc: 0.7417
Fold s2 

...

Fold s3 | Epoch 1 | Train Loss: 0.9517 | Val Loss: 0.6918 | Val Acc: 0.6792
Fold s3 | Epoch 2 | Train Loss: 0.6356 | Val Loss: 0.4321 | Val Acc: 0.8667
Fold s3 | Epoch 3 | Train Loss: 0.4227 | Val Loss: 0.4225 | Val Acc: 0.8375
Fold s3 | Epoch 4 | Train Loss: 0.2473 | Val Loss: 0.4195 | Val Acc: 0.8375
Fold s3 | Epoch 5 | Train Loss: 0.2026 | Val Loss: 0.3616 | Val Acc: 0.8333
Fold s3 | Epoch 6 | Train Loss: 0.1632 | Val Loss: 0.3755 | Val Acc: 0.8542
Fold s3 | Epoch 7 | Train Loss: 0.1753 | Val Loss: 0.9064 | Val Acc: 0.8042
Fold s3 | Epoch 8 | Train Loss: 0.1433 | Val Loss: 0.5666 | Val Acc: 0.8167
Fold s3 | Epoch 9 | Train Loss: 0.1134 | Val Loss: 0.4993 | Val Acc: 0.8875
Fold s3 | Epoch 10 | Train Loss: 0.0737 | Val Loss: 0.5432 | Val Acc: 0.8250
Fold s3 | Epoch 11 | Train Loss: 0.0852 | Val Loss: 0.6150 | Val Acc: 0.8250
Fold s3 | Epoch 12 | Train Loss: 0.0735 | Val Loss: 0.6376 | Val Acc: 0.8458
Fold s3 | Epoch 13 | Train Loss: 0.1367 | Val Loss: 0.5620 | Val Acc: 0.8125
Fold s3 

C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\

Fold s4 | Epoch 1 | Train Loss: 0.9532 | Val Loss: 0.7725 | Val Acc: 0.7625
Fold s4 | Epoch 2 | Train Loss: 0.6261 | Val Loss: 0.5627 | Val Acc: 0.8083
Fold s4 | Epoch 3 | Train Loss: 0.3607 | Val Loss: 0.6029 | Val Acc: 0.7542
Fold s4 | Epoch 4 | Train Loss: 0.2479 | Val Loss: 0.4481 | Val Acc: 0.8208
Fold s4 | Epoch 5 | Train Loss: 0.2138 | Val Loss: 0.4968 | Val Acc: 0.7750
Fold s4 | Epoch 6 | Train Loss: 0.1298 | Val Loss: 0.5880 | Val Acc: 0.7625
Fold s4 | Epoch 7 | Train Loss: 0.1233 | Val Loss: 0.3651 | Val Acc: 0.8292
Fold s4 | Epoch 8 | Train Loss: 0.0617 | Val Loss: 0.6539 | Val Acc: 0.7625
Fold s4 | Epoch 9 | Train Loss: 0.0737 | Val Loss: 0.6626 | Val Acc: 0.7375
Fold s4 | Epoch 10 | Train Loss: 0.0656 | Val Loss: 0.4575 | Val Acc: 0.8125
Fold s4 | Epoch 11 | Train Loss: 0.0514 | Val Loss: 0.6283 | Val Acc: 0.8167
Fold s4 | Epoch 12 | Train Loss: 0.0457 | Val Loss: 0.4433 | Val Acc: 0.8083
Fold s4 | Epoch 13 | Train Loss: 0.0353 | Val Loss: 0.6523 | Val Acc: 0.7750
Fold s4 

C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\4279526295.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_val = torch.tensor(X_val, dtype=torch.float32)
C:\Users\aless\AppData\Local\Temp\ipykernel_24236\

Fold s5 | Epoch 1 | Train Loss: 0.9328 | Val Loss: 0.8115 | Val Acc: 0.5542
Fold s5 | Epoch 2 | Train Loss: 0.6099 | Val Loss: 0.7757 | Val Acc: 0.6000
Fold s5 | Epoch 3 | Train Loss: 0.3527 | Val Loss: 0.8193 | Val Acc: 0.5750
Fold s5 | Epoch 4 | Train Loss: 0.2482 | Val Loss: 0.5494 | Val Acc: 0.6542
Fold s5 | Epoch 5 | Train Loss: 0.2230 | Val Loss: 0.8759 | Val Acc: 0.5708
Fold s5 | Epoch 6 | Train Loss: 0.1875 | Val Loss: 0.8823 | Val Acc: 0.6042
Fold s5 | Epoch 7 | Train Loss: 0.1210 | Val Loss: 1.1806 | Val Acc: 0.5542
Fold s5 | Epoch 8 | Train Loss: 0.1330 | Val Loss: 0.8935 | Val Acc: 0.5958
Fold s5 | Epoch 9 | Train Loss: 0.0795 | Val Loss: 0.8594 | Val Acc: 0.6292
Fold s5 | Epoch 10 | Train Loss: 0.1102 | Val Loss: 1.1450 | Val Acc: 0.6458
Fold s5 | Epoch 11 | Train Loss: 0.1196 | Val Loss: 1.2908 | Val Acc: 0.6042
Fold s5 | Epoch 12 | Train Loss: 0.0995 | Val Loss: 0.7429 | Val Acc: 0.6458
Fold s5 | Epoch 13 | Train Loss: 0.0891 | Val Loss: 1.3141 | Val Acc: 0.6458
Fold s5 